In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
torch.manual_seed(42)

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [4]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [5]:
from torchvision import datasets

train_dataset = datasets.ImageFolder(root=(r"C:\Users\utkar\1project\dataset\train"), transform=train_transform)

In [6]:
test_dataset = datasets.ImageFolder(root=(r"C:\Users\utkar\1project\dataset\test"), transform=val_test_transform)

val_dataset = datasets.ImageFolder(root=(r"C:\Users\utkar\1project\dataset\val"), transform=val_test_transform)

In [7]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32
NUM_WORKERS = 6

In [8]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,num_workers=NUM_WORKERS, pin_memory=True)

In [9]:
import torch
import torch.nn as nn

class LungNN(nn.Module):
    def __init__(self, input_features=3):
        super().__init__()

        self.features = nn.Sequential(
            # 3 to 32 channel
            nn.Conv2d(input_features, 32, kernel_size=3, padding='same'),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), #output 32@112x112

            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), #output 64@56x56

            nn.Conv2d(64, 128, kernel_size=3, padding='same'),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # output 128@28x28

            nn.Conv2d(128, 256, kernel_size=3, padding='same'),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # output 256@14x14

            nn.Conv2d(256, 512, kernel_size=3, padding='same'),
            nn.BatchNorm2d(512), 
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)  # output 512@7x7
        )
        
        # Classifier 
        # 512 channel * 7 * 7 spatial resolution = 25,088 features
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            nn.Linear(64, 5) # 5 lung disease classes
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Instantiate and push to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LungNN(input_features=3).to(device)

print("Architecture successfully initialized on GPU!")

Architecture successfully initialized on GPU!


In [10]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print("Optimizer updated for the new model design.")

Optimizer updated for the new model design.


In [11]:
import time

epochs = 50

for epoch in range(epochs):
    epoch_start_time = time.time()
    
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()#reset gredient
        

        outputs = model(images)#forward
        loss = criterion(outputs, labels)

        
        loss.backward()#backward
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)# Metric tracking calculations
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    epoch_train_loss = running_loss / len(train_loader.dataset)
    epoch_train_acc = (correct_train / total_train) * 100
    

    model.eval() #validation
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    # Turn off gradient engine to conserve VRAM
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    epoch_val_loss = val_loss / len(val_loader.dataset)
    epoch_val_acc = (correct_val / total_val) * 100
    
    epoch_duration = time.time() - epoch_start_time
    
    # Print metrics for the current epoch iteration
    print(f"Epoch [{epoch+1}/{epochs}] ({epoch_duration:.1f}s) -> "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")

Epoch [1/50] (56.5s) -> Train Loss: 1.0172 | Train Acc: 57.27% | Val Loss: 0.6658 | Val Acc: 70.73%
Epoch [2/50] (54.6s) -> Train Loss: 0.7389 | Train Acc: 68.70% | Val Loss: 0.5913 | Val Acc: 75.25%
Epoch [3/50] (55.1s) -> Train Loss: 0.6451 | Train Acc: 72.61% | Val Loss: 0.5715 | Val Acc: 76.29%
Epoch [4/50] (55.3s) -> Train Loss: 0.6007 | Train Acc: 74.33% | Val Loss: 0.5161 | Val Acc: 78.72%
Epoch [5/50] (53.7s) -> Train Loss: 0.5695 | Train Acc: 75.77% | Val Loss: 0.5168 | Val Acc: 79.27%
Epoch [6/50] (54.0s) -> Train Loss: 0.5459 | Train Acc: 77.32% | Val Loss: 0.4664 | Val Acc: 81.15%
Epoch [7/50] (54.5s) -> Train Loss: 0.5109 | Train Acc: 77.98% | Val Loss: 0.4941 | Val Acc: 80.41%
Epoch [8/50] (58.9s) -> Train Loss: 0.4927 | Train Acc: 79.35% | Val Loss: 0.4775 | Val Acc: 80.31%
Epoch [9/50] (56.6s) -> Train Loss: 0.5004 | Train Acc: 78.41% | Val Loss: 0.4428 | Val Acc: 81.45%
Epoch [10/50] (55.3s) -> Train Loss: 0.4740 | Train Acc: 79.63% | Val Loss: 0.4029 | Val Acc: 83.13%

In [12]:
import torch

MODEL_PATH = "LungNN2.pth"

# Save the state_dict (weights)
torch.save(model.state_dict(), MODEL_PATH)

print(f"Successfully saved weights to: {MODEL_PATH}")

Successfully saved weights to: LungNN2.pth


In [18]:
model.eval()  
test_loss = 0.0
correct_test = 0
total_test = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()

final_test_loss = test_loss / len(test_loader.dataset)
final_test_acc = (correct_test / total_test) * 100

print("Results")
print(f"Test Loss: {final_test_loss:.2f}")
print(f"Test Accuracy: {final_test_acc:.2f}%")



Results
Test Loss: 0.39
Test Accuracy: 83.75%


In [19]:
'''import os
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt'''


In [20]:
'''class_names = ['Normal', 'Viral pneumino']

# 2. Select hardware target
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Inference pipeline initialized using: {device}")'''

Inference pipeline initialized using: cuda
